# NUMARI
**Reglas:** Selecciona un número → camina exactamente esa cantidad de celdas vacías (↑↓←→) → selecciona el siguiente número adyacente → repite hasta llenar toda la grilla.

> Escribe **q** en el campo de texto para volver al menú.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import copy, time

LEVELS = [
    {
        "name": "Nivel 1 — 4x4",
        "grid": [
            [3, 0, 0, 0],
            [0, 0, 0, 3],
            [3, 0, 0, 0],
            [0, 0, 0, 3],
        ]
    },
    {
        "name": "Nivel 2 — 4x4",
        "grid": [
            [3, 0, 3, 0],
            [0, 0, 0, 0],
            [0, 0, 0, 0],
            [0, 3, 0, 3],
        ]
    },
    {
        "name": "Nivel 3 — 4x4",
        "grid": [
            [3, 0, 0, 0],
            [2, 0, 0, 2],
            [0, 0, 1, 0],
            [0, 0, 3, 0],
        ]
    },
    {
        "name": "Nivel 4 — 5x5",
        "grid": [
            [4, 0, 0, 0, 0],
            [0, 0, 0, 0, 4],
            [4, 0, 0, 0, 0],
            [0, 0, 0, 0, 4],
            [4, 0, 0, 0, 0],
        ]
    },
    {
        "name": "Nivel 5 — 5x5",
        "grid": [
            [4, 0, 4, 0, 4],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 0, 0, 0, 0],
            [0, 4, 0, 4, 0],
        ]
    },
]

class Cell:
    def __init__(self, v):
        self.value   = v
        self.visited = False
    def is_num(self):   return self.value > 0
    def is_empty(self): return self.value == 0

class Game:
    DIRS = [(-1,0),(1,0),(0,-1),(0,1)]

    def __init__(self, level):
        raw          = level["grid"]
        self.rows    = len(raw)
        self.cols    = len(raw[0])
        self.total   = self.rows * self.cols
        self.board   = [[Cell(raw[r][c]) for c in range(self.cols)] for r in range(self.rows)]
        self.visited = 0
        self.history = []
        self.phase   = "SELECT"
        self.pos     = None
        self.steps   = 0
        self.won     = False
        self.stuck   = False
        self.msg     = "Selecciona un numero para empezar"
        self.ok      = True

    def neighbors(self, r, c):
        return [(r+dr, c+dc) for dr,dc in self.DIRS
                if 0 <= r+dr < self.rows and 0 <= c+dc < self.cols]

    def save(self):
        self.history.append({
            "phase": self.phase, "pos": self.pos, "steps": self.steps,
            "visited": self.visited,
            "v": [[self.board[r][c].visited for c in range(self.cols)] for r in range(self.rows)],
        })

    def undo(self):
        if not self.history:
            self.msg, self.ok = "Nada que deshacer", False
            return
        s = self.history.pop()
        self.phase, self.pos, self.steps, self.visited = s["phase"], s["pos"], s["steps"], s["visited"]
        self.won   = False
        self.stuck = False
        for r in range(self.rows):
            for c in range(self.cols):
                self.board[r][c].visited = s["v"][r][c]
        self.msg, self.ok = "Deshecho", True

    def select(self, r, c):
        if self.won or self.stuck: return
        cell = self.board[r][c]
        if cell.visited:
            self.msg, self.ok = "Ya visitada", False
            return
        self.save()

        if self.phase == "SELECT":
            if not cell.is_num():
                self.history.pop()
                self.msg, self.ok = "Debe ser un numero", False
                return
            cell.visited = True
            self.visited += 1
            self.pos   = (r, c)
            self.steps = cell.value
            self.phase = "WALK"
            self.msg, self.ok = f"Numero {cell.value} — camina {cell.value} celda(s) vacia(s)", True

        else:
            pr, pc = self.pos
            if (r, c) not in self.neighbors(pr, pc):
                self.history.pop()
                self.msg, self.ok = "Solo celdas adyacentes", False
                return
            if self.steps > 0:
                if not cell.is_empty():
                    self.history.pop()
                    self.msg, self.ok = f"Faltan {self.steps} celda(s) vacia(s)", False
                    return
                cell.visited = True
                self.visited += 1
                self.pos    = (r, c)
                self.steps -= 1
                if self.steps == 0:
                    self.msg, self.ok = "Selecciona el siguiente numero adyacente", True
                else:
                    self.msg, self.ok = f"Quedan {self.steps} paso(s)", True
            else:
                if not cell.is_num():
                    self.history.pop()
                    self.msg, self.ok = "Selecciona un numero adyacente", False
                    return
                cell.visited = True
                self.visited += 1
                self.pos   = (r, c)
                self.steps = cell.value
                self.msg, self.ok = f"Numero {cell.value} — camina {cell.value} celda(s) vacia(s)", True

        if self.visited == self.total:
            self.won = True
            self.msg, self.ok = "GANASTE", True
            return

        # Si no hay movimientos validos y no gano → atascado
        if self.pos is not None and not self.valid_moves():
            self.stuck = True
            self.msg, self.ok = "Sin salida — ruta descartada. Deshaz o reinicia.", False

    def valid_moves(self):
        if self.won or self.pos is None: return set()
        r, c = self.pos
        out  = set()
        for nr, nc in self.neighbors(r, c):
            cell = self.board[nr][nc]
            if cell.visited: continue
            if self.steps > 0 and cell.is_empty(): out.add((nr, nc))
            if self.steps == 0 and cell.is_num():  out.add((nr, nc))
        return out


class NumariUI:
    C_NUM = "#f5c518"
    C_VAL = "#5dade2"
    C_VNM = "#58d68d"
    C_VIS = "#27ae60"
    C_CUR = "#f39c12"
    C_OFF = "#2a2d3e"

    def __init__(self):
        self.idx      = 0
        self.in_menu  = True
        self.out      = widgets.Output()
        display(self.out)
        self._show_menu()

    # ── Menu principal ───────────────────────────────────────────────────────

    def _show_menu(self):
        self.in_menu = True
        btns = []
        for i, lv in enumerate(LEVELS):
            b = widgets.Button(
                description=lv["name"],
                button_style="info",
                layout=widgets.Layout(width="200px", height="34px", margin="3px 0"))
            b.on_click(lambda _, i=i: self._start(i))
            btns.append(b)

        title = widgets.HTML(
            '<b style="font-family:monospace;color:#5dade2;font-size:20px;'
            'letter-spacing:4px">NUMARI</b>'
            '<p style="color:#888;font-family:monospace;font-size:12px;margin:4px 0">'
            'Selecciona un nivel</p>')

        menu = widgets.VBox(
            [title] + btns,
            layout=widgets.Layout(padding="16px", align_items="flex-start"))

        with self.out:
            clear_output(wait=True)
            display(menu)

    # ── Juego ────────────────────────────────────────────────────────────────

    def _start(self, idx):
        self.idx     = idx
        self.in_menu = False
        self.game    = Game(LEVELS[idx])
        self._build_game()
        self._draw()
        with self.out:
            clear_output(wait=True)
            display(self.root)

    def _build_game(self):
        self.header  = widgets.HTML()
        self.grid    = widgets.VBox()
        self.msg_box = widgets.HTML()

        self.b_undo = widgets.Button(
            description="Deshacer", button_style="warning",
            layout=widgets.Layout(height="32px", width="105px"))

        self.b_quit = widgets.Button(
            description="Salir (Q)", button_style="danger",
            layout=widgets.Layout(height="32px", width="105px"))

        self.cmd_input = widgets.Text(
            placeholder="Escribe q para salir",
            layout=widgets.Layout(width="160px", height="32px"))

        self.b_undo.on_click(lambda _: (self.game.undo(), self._draw()))
        self.b_quit.on_click(lambda _: self._show_menu())
        self.cmd_input.on_submit(self._handle_key)

        controls = widgets.HBox(
            [self.b_undo, self.b_quit, self.cmd_input],
            layout=widgets.Layout(gap="6px", margin="6px 0"))

        self.root = widgets.VBox(
            [self.header, self.grid, self.msg_box, controls],
            layout=widgets.Layout(padding="14px", max_width="500px", align_items="flex-start"))

    def _handle_key(self, widget):
        if widget.value.strip().lower() == "q":
            self._show_menu()
        widget.value = ""

    def _draw(self):
        g     = self.game
        lv    = LEVELS[self.idx]
        pct   = int(g.visited / g.total * 100)
        valid = g.valid_moves()

        self.header.value = (
            f'<div style="font-family:monospace;margin-bottom:6px">'
            f'<b style="color:#5dade2;font-size:18px;letter-spacing:4px">NUMARI</b>'
            f'&nbsp;&nbsp;<span style="color:#888;font-size:12px">'
            f'{lv["name"]} &nbsp;|&nbsp; {g.visited}/{g.total} celdas</span>'
            f'<div style="margin-top:4px;background:#222;border-radius:3px;height:4px;width:220px">'
            f'<div style="background:#5dade2;border-radius:3px;height:4px;width:{pct}%"></div></div>'
            f'</div>'
        )

        rows = []
        for r in range(g.rows):
            row = []
            for c in range(g.cols):
                row.append(self._btn(g, r, c, valid))
            rows.append(widgets.HBox(row, layout=widgets.Layout(gap="3px")))
        self.grid.children = rows
        self.grid.layout   = widgets.Layout(gap="3px", margin="6px 0")

        if g.won:
            bg, tc = "#0d2e18", "#2ecc71"
        elif g.stuck:
            bg, tc = "#2e0d0d", "#e74c3c"
        elif g.ok:
            bg, tc = "#0d2340", "#5dade2"
        else:
            bg, tc = "#2e1a00", "#f39c12"

        self.msg_box.value = (
            f'<div style="background:{bg};color:{tc};padding:6px 12px;'
            f'border-radius:5px;font-family:monospace;font-size:13px;margin:2px 0">'
            f'{g.msg}</div>')

    def _btn(self, g, r, c, valid):
        cell       = g.board[r][c]
        is_current = (g.pos == (r, c))
        is_valid   = (r, c) in valid

        if cell.visited:
            if is_current: bg, tc, lbl = "#3d1f00", self.C_CUR, "★"
            elif cell.is_num(): bg, tc, lbl = "#0d2e18", self.C_VIS, str(cell.value)
            else:               bg, tc, lbl = "#0d2e18", self.C_VIS, "·"
        elif is_valid:
            if cell.is_num(): bg, tc, lbl = "#0d2e18", self.C_VNM, str(cell.value)
            else:             bg, tc, lbl = "#0d2340", self.C_VAL, "·"
        elif cell.is_num():   bg, tc, lbl = "#2a2510", self.C_NUM, str(cell.value)
        else:                  bg, tc, lbl = self.C_OFF, "#3a4060", "·"

        sz  = "50px" if g.cols <= 4 else "42px"
        btn = widgets.Button(
            description=lbl,
            layout=widgets.Layout(width=sz, height=sz),
            disabled=(cell.visited and not is_current) or g.won or g.stuck)
        btn.style.button_color = bg
        btn.style.font_weight  = "bold" if (cell.is_num() or is_current) else "normal"

        def click(b, row=r, col=c):
            self.game.select(row, col)
            self._draw()

        btn.on_click(click)
        return btn


NumariUI()
